# Swarm Node State Visualization

Parse `swarm.log` and plot **incorporated**, **pending**, and **wanted** over time, one line per node.

In [ ]:
import re
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
LOG_FILE = "../swarm.log"

LINE_RE = re.compile(
    r'time=(\S+)\s+level=\S+\s+msg="received message"\s+'
    r'node=([0-9a-f]+)\s+ref=[0-9a-f]+\s+'
    r'incorporated=(\d+)\s+pending=(\d+)\s+wanted=(\d+)'
)

In [ ]:
def parse_log(path):
    """Return {node: [(seconds_from_start, incorporated, pending, wanted), ...]}."""
    records = defaultdict(list)
    t0 = None
    with open(path) as f:
        for line in f:
            m = LINE_RE.search(line)
            if not m:
                continue
            ts_str, node, inc, pend, want = m.groups()
            ts = datetime.fromisoformat(ts_str)
            if t0 is None:
                t0 = ts
            elapsed = (ts - t0).total_seconds()
            records[node].append((elapsed, int(inc), int(pend), int(want)))
    return records

records = parse_log(LOG_FILE)
print(f"Parsed {sum(len(v) for v in records.values())} entries across {len(records)} nodes")

In [ ]:
fields = ["incorporated", "pending", "wanted"]
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, field, idx in zip(axes, fields, [1, 2, 3]):
    for node in sorted(records):
        data = records[node]
        t = [r[0] for r in data]
        v = [r[idx] for r in data]
        ax.plot(t, v, label=node, linewidth=0.7, alpha=0.85)
    ax.set_ylabel(field)
    ax.legend(fontsize=7, ncol=5, loc="upper left")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("time (seconds from start)")
axes[0].set_title("Swarm node state over time")
fig.tight_layout()